# Lab: Sequence Modeling with RNN, LSTM, and GRU

## 1. Introduction: The "Memory" Problem

In standard Feed-Forward Neural Networks (MLP), inputs are independent. If you put in an image of a cat, it tells you "cat". It doesn't care what the previous image was.

**But language is sequential.** To understand "*The movie was not good*", you need to know "*not*" came before "*good*".

In this lab, we will:
1.  Train a model to generate text (Character-Level generation).
2.  Start with a **Vanilla RNN** (Recurrent Neural Network).
3.  Upgrade to **LSTM** (Long Short-Term Memory) and **GRU** (Gated Recurrent Unit) to see how they handle memory better.
4.  Use a scientific/academic definition as our training data.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Reproducibility
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. The Data: Academic Definition

We use a text about **Deep Learning** from Wikipedia. The model will learn to write like a textbook!

In [ ]:
text_data = """
Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning. 
Learning can be supervised, semi-supervised or unsupervised. 
Deep-learning architectures such as deep neural networks, deep belief networks, deep reinforcement learning, recurrent neural networks, 
convolutional neural networks and transformers have been applied to fields including computer vision, speech recognition, natural language processing, machine translation, bioinformatics, drug design, medical image analysis, climate science, material inspection and board game programs, where they have produced results comparable to and in some cases surpassing human expert performance. 
Artificial neural networks (ANNs) were inspired by information processing and distributed communication nodes in biological systems. 
ANNs have various differences from biological brains. Specifically, artificial neural networks tend to be static and symbolic, while the biological brain of most living organisms is dynamic (plastic) and analog.
"""

print(f"Text Length: {len(text_data)} characters")
print(f"Sample: {text_data[:100]}...")

### 3. Preprocessing: Converting Text to Numbers

Neural Networks need numbers (Tensors), not strings. We create a dictionary.

In [ ]:
# 1. Create Vocabulary
chars = sorted(list(set(text_data)))
vocab_size = len(chars)

# 2. Mappings
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }

print(f"Vocabulary Size: {vocab_size} unique characters")
print(f"Chars: {''.join(chars)}")

# 3. Helper Functions
def string_to_tensor(text):
    """Converts a string to a tensor of indices."""
    indices = [char_to_ix[ch] for ch in text]
    return torch.tensor(indices, dtype=torch.long).to(device)

def tensor_to_string(tensor):
    # tensor shape [Batch, Seq] or [Seq]
    if tensor.dim() == 2:
        tensor = tensor[0] # take first batch if batched
    return "".join([ix_to_char[idx.item()] for idx in tensor])

# Test
print(f"Test 'Deep': {string_to_tensor('Deep')}")

## 4. The Unified Model (RNN / LSTM / GRU)

We create a single class that can behave as any of the three models based on a parameter.

### Key Concepts:

1.  **Embedding Layer**: Converts an integer (index `5`) into a dense vector (e.g., `[0.1, -0.5, ...]`)
2.  **RNN/LSTM/GRU Layer**: The core sequence processor.
    *   **RNN**: Computes $h_t = \tanh(W_x x_t + W_h h_{t-1})$
    *   **LSTM**: Adds a "Cell State" $C_t$ (Long term memory) and 3 Gates (Input, Forget, Output).
    *   **GRU**: Simplified LSTM, merges gates into Update & Reset.
3.  **Linear Layer**: Decodes the hidden state back to vocabulary size logic (prediction).

In [ ]:
class SequenceModel(nn.Module):
    def __init__(self, model_type, input_size, hidden_size, output_size, n_layers=1):
        super(SequenceModel, self).__init__()
        self.model_type = model_type.lower()
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        
        # 1. Embedding
        self.embedding = nn.Embedding(input_size, hidden_size)
        
        # 2. Recurrent Layer (Choice)
        if self.model_type == 'rnn':
            self.rnn = nn.RNN(hidden_size, hidden_size, n_layers, batch_first=True)
        elif self.model_type == 'lstm':
            self.rnn = nn.LSTM(hidden_size, hidden_size, n_layers, batch_first=True)
        elif self.model_type == 'gru':
            self.rnn = nn.GRU(hidden_size, hidden_size, n_layers, batch_first=True)
        
        # 3. Output Layer
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x, hidden):
        # x shape: [Batch, Seq]
        embed = self.embedding(x)
        
        # Run RNN/LSTM/GRU
        # out shape: [Batch, Seq, Hidden]
        # hidden shape: depends on model type
        out, hidden = self.rnn(embed, hidden)
        
        # Flatten for classification
        out = out.reshape(-1, self.hidden_size)
        out = self.fc(out)
        return out, hidden
    
    def init_hidden(self, batch_size):
        weight = next(self.parameters()).data
        
        if self.model_type == 'lstm':
            # LSTM needs TWO hidden states: (Hidden, Cell)
            return (weight.new(self.n_layers, batch_size, self.hidden_size).zero_(),
                    weight.new(self.n_layers, batch_size, self.hidden_size).zero_())
        else:
            # RNN and GRU need ONE hidden state
            return weight.new(self.n_layers, batch_size, self.hidden_size).zero_()

## 5. Text Generation Function
We use the model to generate text character by character.

In [ ]:
def generate_text(model, start_str="Deep", predict_len=100, temperature=0.8):
    model.eval()
    hidden = model.init_hidden(1)
    
    # Process the start string
    input_seq = string_to_tensor(start_str).unsqueeze(0)
    
    # Warm up hidden state
    for i in range(len(start_str) - 1):
        _, hidden = model(input_seq[:, i].unsqueeze(1), hidden)
    
    # Start predicting
    inp = input_seq[:, -1].unsqueeze(1)
    predicted_text = start_str
    
    for i in range(predict_len):
        output, hidden = model(inp, hidden)
        
        # Temperature sampling
        output_dist = output.data.view(-1).div(temperature).exp()
        top_i = torch.multinomial(output_dist, 1)[0]
        
        predicted_char = ix_to_char[top_i.item()]
        predicted_text += predicted_char
        
        inp = string_to_tensor(predicted_char).unsqueeze(0)
        
    return predicted_text

## 6. Training Loop

We will train three instances: `rnn_model`, `lstm_model`, and `gru_model`.

In [ ]:
def get_batch(chunk_len=50, batch_size=16):
    input_batch = []
    target_batch = []
    
    for _ in range(batch_size):
        start_idx = np.random.randint(0, len(text_data) - chunk_len - 1)
        chunk = text_data[start_idx : start_idx + chunk_len + 1]
        input_batch.append(string_to_tensor(chunk[:-1]))
        target_batch.append(string_to_tensor(chunk[1:]))
        
    return torch.stack(input_batch), torch.stack(target_batch)

def train(model, n_epochs=180):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    loss_history = []
    
    model.train()
    print(f"--- Training {model.model_type.upper()} ---")
    
    for epoch in range(1, n_epochs + 1):
        hidden = model.init_hidden(16) # batch size 16
        inputs, targets = get_batch()
        
        model.zero_grad()
        
        # Detach hidden state to prevent backprop through entire history
        if model.model_type == 'lstm':
            hidden = (hidden[0].detach(), hidden[1].detach())
        else:
            hidden = hidden.detach()
            
        output, hidden = model(inputs, hidden)
        loss = criterion(output, targets.view(-1))
        loss.backward()
        optimizer.step()
        
        loss_history.append(loss.item())
        
        if epoch % 100 == 0:
            print(f"Epoch {epoch} | Loss: {loss.item():.4f}")
            
    return loss_history

## 7. Experiments and Visualization

In [ ]:
# Common Hyperparameters
HIDDEN_SIZE = 128
N_LAYERS = 1
EPOCHS = 100

# 1. Train Vanilla RNN
rnn_model = SequenceModel('rnn', vocab_size, HIDDEN_SIZE, vocab_size, N_LAYERS).to(device)
rnn_losses = train(rnn_model, n_epochs=EPOCHS)

# 2. Train LSTM
lstm_model = SequenceModel('lstm', vocab_size, HIDDEN_SIZE, vocab_size, N_LAYERS).to(device)
lstm_losses = train(lstm_model, n_epochs=EPOCHS)

# 3. Train GRU
gru_model = SequenceModel('gru', vocab_size, HIDDEN_SIZE, vocab_size, N_LAYERS).to(device)
gru_losses = train(gru_model, n_epochs=EPOCHS)

### Compare Loss Curves

Typically, LSTM and GRU stabilize faster and reach lower loss than Vanilla RNN on complex data. On this small dataset, they might look similar, but notice if RNN fluctuates more.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(rnn_losses, label='Vanilla RNN', alpha=0.7)
plt.plot(lstm_losses, label='LSTM', alpha=0.7)
plt.plot(gru_losses, label='GRU', alpha=0.7)
plt.title("Training Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Generate Text Samples

In [ ]:
print("Generate Text: Vanilla RNN")
print(generate_text(rnn_model, start_str="Artificial", predict_len=200))

print("\nGenerate Text: LSTM")
print(generate_text(lstm_model, start_str="Network", predict_len=200))

print("\nGenerate Text: GRU")
print(generate_text(gru_model, start_str="ANN", predict_len=200))

## 8. Summary: What we learned

1.  **RNN**: Simple, but suffers from "Vanishing Gradient" (forgets long context).
2.  **LSTM**: Adds **Cell State** and **Gates** to selectively remember/forget. (Slower but powerful).
3.  **GRU**: Simplified LSTM (merged gates). Often faster and just as good.
4.  **Generation**: All models learned to spell words like "network" and "learning" from character-level input alone!